# Check an Analyst report

Copy this file into `private/reviews/` and rename the copy to `<packet_id>_review.ipynb`
(for example, `EP_0003_review.ipynb`). Work in that copy; keep this template blank.

1. Set `attempt_id` below to the folder name printed by generation.
2. Run the cells and read each claim beside its evidence.
3. Fill in **Notes / feedback** and make any fixes in the edit cell.
4. When finished, set `ready = True`, run all cells again, and save the notebook.

The last cell saves the final report to public. The original candidate stays unchanged.
There are no model calls here. Leave `ready = False` while checking.


In [ ]:
import copy
import json
import sys
from decimal import Decimal
from pathlib import Path
from pprint import pprint

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/benchmark/private/packets").is_dir())
sys.path.insert(0, str(repo))
from src.benchmark.prepare import read, save, mask, assert_masked, check_report
from src.benchmark.schemas import Packet, Report

folder = repo / "data/benchmark"
attempt_id = "<attempt-id>"  # change this in your copy
attempt_folder = folder / "private/attempts" / attempt_id
attempt = read(attempt_folder / "attempt.json")
sent_packet = json.loads(attempt["request"]["messages"][1]["content"])["evidence_packet"]
original_report = read(attempt_folder / "candidate.json")
aliases = read(folder / "private/aliases.json")
packet = mask(sent_packet, aliases)
report = copy.deepcopy(original_report)
print("Packet:", packet["packet_id"], "| Model:", attempt["request"]["model"])
print("Packet masking changed:", packet != sent_packet)

In [ ]:
print("Recommendation:", report["recommendation"])
evidence = {e["evidence_id"]: e["text"] for e in packet["evidence"]}
for claim in report["claims"]:
    print("\nCLAIM", claim["claim_id"], claim["text"])
    for evidence_id in claim["evidence_ids"]:
        print("EVIDENCE", evidence_id, evidence.get(evidence_id, "MISSING"))
print("\nRationale:", report["rationale"])

## Numbers

Compare the claimed amount or percentage with the calculation. Also read the prose above:
these checks only cover the structured assertion, not every number in a sentence.

In [ ]:
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        print(claim["claim_id"], "Check against the excerpt above.")
        continue
    facts = [f for f in packet["facts"] if f["metric"] == spec["metric"]
             and f["evidence_id"] in claim["evidence_ids"]]
    periods = [spec["period"]] if spec["kind"] == "numeric" else [spec["from_period"], spec["to_period"]]
    matches = [[f for f in facts if f["period"] == period] for period in periods]
    if any(len(found) != 1 for found in matches):
        print(claim["claim_id"], "No exact fact match; check manually.")
        continue
    values = [Decimal(found[0]["value"]) for found in matches]
    if spec["kind"] == "percentage_change" and values[0] == 0:
        print(claim["claim_id"], "Cannot calculate percentage change from zero.")
        continue
    expected = values[0] if spec["kind"] == "numeric" else (values[1] - values[0]) / values[0] * 100
    print(claim["claim_id"], "claimed:", spec["value"], "calculated:", expected,
          "rounded:", expected.quantize(Decimal("0.01")))
pprint(check_report(Packet.model_validate(packet), Report.model_validate(report)))

## Notes / feedback

Replace the blanks with short notes. Use the evidence-checking notebook to check the source PDF too.

- Source excerpts and masking: ...
- Claims, numbers, years, units and citations: ...
- Reasoning and risks / counter-evidence: ...
- Problems found and what I changed (or no changes needed): ...
- Decision: not checked yet.


In [ ]:
# Make any corrections here. Skip this if no changes are needed.
# Example: report["claims"][0]["text"] = "Corrected claim..."
# If you correct a number, update its check_spec too, if present.
# Keep the edits in this cell so running the notebook again applies them.

pprint(report, sort_dicts=False)


## Save the final copy

Set `ready = True` only when the checks and edits are finished.
This checks the edited report and saves it with the masked packet to `public/`.
It replaces existing files with the same IDs, so use this for the working dataset.
Keep released dataset versions unchanged.

Save this notebook with its outputs. These notes are also the feedback;
there is no separate review JSON or feedback file to fill in.


In [ ]:
ready = False
if ready:
    final_report = Report.model_validate(report)
    final_packet = Packet.model_validate(packet)
    checks = check_report(final_packet, final_report)
    assert not checks["errors"], checks["errors"]
    assert_masked(report, aliases)
    assert_masked(packet, aliases)
    save(folder / "public/evidence_packets" / (packet["packet_id"] + ".json"), final_packet.model_dump(mode="json"))
    output = folder / "public/reports" / (report["report_id"] + ".json")
    save(output, final_report.model_dump(mode="json"))
    print("Saved:", output)
else:
    print("Not saved yet.")